# T4.2 — Particionamento por grupo

Este notebook usa a planilha de decisões da T3.3 e o dicionário da T3.4 para criar um **manifesto provisório** de treino, validação e teste. Nenhuma imagem é copiada. Os caminhos no manifesto apontam para os arquivos de origem, que podem estar no Drive.

**Regra principal:** imagens da mesma origem, duplicatas exatas e pares visuais confirmados ficam na mesma partição. Pares classificados como falsos positivos não são unidos. Um arquivo opcional `assets/data/vinculos_mesma_planta.csv`, com colunas `image_id_1,image_id_2`, permite acrescentar vínculos de planta ou sessão identificados manualmente.

As 55 ocorrências da T3.4 recebem seus IDs `AGRO-*`. As ocorrências de `dataset` recebem IDs candidatos `DATASET-*`. **O notebook não funde rótulos por semelhança de nome ou agente causal**: essa decisão depende de aprovação agronômica. Também não certifica independência entre fotos de catálogo sem metadados de planta ou captura.

Dependências: `pandas`, `openpyxl` e Jupyter. Execute as células em ordem, a partir de qualquer diretório dentro do repositório.

In [1]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display

ROOT = next(
    p for p in (Path.cwd(), *Path.cwd().parents)
    if (p / "assets/data/diagnostico_qualidade_T3_3_revisado.xlsx").exists()
)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.particionamento_t4_2 import (
    carregar_dados, construir_grupos, separar_grupos, salvar_resultados
)

SEMENTE = 42
VINCULOS = ROOT / "assets/data/vinculos_mesma_planta.csv"
print("Repositório:", ROOT)
print("Vínculos manuais:", "presentes" if VINCULOS.exists() else "ausentes")

Repositório: C:\Users\Inteli\Desktop\2026_02_AgroApp
Vínculos manuais: ausentes


## 1. Seleção e rótulos candidatos

Somente linhas com `decisao_final = manter` entram na divisão. O dicionário T3.4 associa os registros dos acervos antigos por **cultura + rótulo de origem**. Os `class_id` de `dataset` continuam separados até que haja uma tabela aprovada de equivalência e fusão de classes. A contagem de ocorrências do diagnóstico inclui imagens excluídas; por isso pode diferir da quantidade de rótulos candidatos selecionados aqui.

In [2]:
mantidas, pares_visuais, proveniencia = carregar_dados(ROOT)
print("Registros do diagnóstico:", proveniencia["total_diagnostico"])
print("Imagens selecionadas:", proveniencia["total_mantidas"])
print("Classes candidatas entre as selecionadas:", proveniencia["classes_candidatas"])
display(
    mantidas.groupby(["fonte", "classe_candidata"], dropna=False)
    .size().rename("imagens").reset_index().head(12)
)

Registros do diagnóstico: 3460
Imagens selecionadas: 3262
Classes candidatas entre as selecionadas: 64


,fonte,classe_candidata,imagens
0,acervo_milho,AGRO-001,5
1,acervo_milho,AGRO-002,2
2,acervo_milho,AGRO-003,4
3,acervo_milho,AGRO-004,4
4,acervo_soja,AGRO-005,2
5,acervo_soja,AGRO-006,2
6,acervo_soja,AGRO-008,2
7,acervo_soja,AGRO-009,3
8,acervo_soja,AGRO-010,2
9,acervo_soja,AGRO-011,3


## 2. Construção dos grupos indivisíveis

O agrupamento usa componentes conectados: se A compartilha origem com B e B é duplicata confirmada de C, os três ficam juntos. `grupo_origem` é uma pista de origem; para imagens de catálogo e dos acervos antigos ele pode ser apenas um identificador por arquivo. Complete `vinculos_mesma_planta.csv` quando a equipe identificar fotografias da mesma planta ou sessão que ainda não compartilham grupo. **Sem essa confirmação, o resultado continua provisório.**

In [3]:
agrupadas, diagnostico_grupos = construir_grupos(mantidas, pares_visuais, VINCULOS)
display(pd.DataFrame([diagnostico_grupos]))
display(
    agrupadas.groupby("grupo_particao").size()
    .rename("imagens_por_grupo").describe().to_frame()
)

,grupos_independentes_provisorios,pares_visuais_confirmados,pares_visuais_falsos,vinculos_manuais,vinculos_manuais_sha256
0,2398,95,4,0,None


,imagens_por_grupo
count,2398.000000
mean,1.360300
std,1.112882
min,1.000000
25%,1.000000
50%,1.000000
75%,1.000000
max,17.000000


## 3. Distribuição dos grupos

As proporções **70%/15%/15%** são alvos, não cotas impostas a cada classe. Classes com menos de três grupos independentes ficam apenas no treino, com status exploratório; não serão apresentadas como avaliadas em validação e teste. Para as demais, a rotina reserva grupos para os três conjuntos e distribui os restantes buscando equilíbrio de imagens, rótulos e fontes. O agrupamento sempre prevalece sobre a estratificação.

In [4]:
manifesto, resumo, auditoria = separar_grupos(agrupadas, semente=SEMENTE)
display(pd.DataFrame({
    "particao": list(auditoria["imagens_por_particao"]),
    "imagens": list(auditoria["imagens_por_particao"].values()),
    "grupos": list(auditoria["grupos_por_particao"].values()),
}))
print("Classes exploratórias (<3 grupos):", auditoria["classes_exploratorias"])
print("Classes candidatas à avaliação:", auditoria["classes_candidatas_a_avaliacao"])
print("Candidatas sem cobertura completa:", auditoria["classes_candidatas_sem_cobertura"])
print("Pendências da alocação:", auditoria["pendencias_da_ancoragem"])
display(resumo.sort_values(["grupos", "classe_candidata"]).head(15))

,particao,imagens,grupos
0,treino,2287,1666
1,validacao,488,366
2,teste,487,366


Classes exploratórias (<3 grupos): 36
Classes candidatas à avaliação: 28
Candidatas sem cobertura completa: []
Pendências da alocação: []


,classe_candidata,ocorrencia,status_classe,imagens,grupos,imagens_treino,grupos_treino,imagens_validacao,grupos_validacao,imagens_teste,grupos_teste,cobertura_tres_particoes
10,AGRO-012,Fungo de pós colheita,exploratoria_menos_de_3_grupos,1,1,1,1,0,0,0,0,False
11,AGRO-014,"Fungo-de-pós-colheita, Podridão-dos-grãos-arma...",exploratoria_menos_de_3_grupos,1,1,1,1,0,0,0,0,False
18,AGRO-021,Mela ou requeima,exploratoria_menos_de_3_grupos,1,1,1,1,0,0,0,0,False
20,AGRO-023,"Mofo-branco, Podridão-de-sclerotinia",exploratoria_menos_de_3_grupos,1,1,1,1,0,0,0,0,False
28,AGRO-032,Nematóide-reniforme,exploratoria_menos_de_3_grupos,1,1,1,1,0,0,0,0,False
30,AGRO-034,Phomopsis-da-semente,exploratoria_menos_de_3_grupos,1,1,1,1,0,0,0,0,False
31,AGRO-035,"Podridão da semente, Podridão de Fusarium",exploratoria_menos_de_3_grupos,1,1,1,1,0,0,0,0,False
34,AGRO-038,Podridão-de-raízes,exploratoria_menos_de_3_grupos,1,1,1,1,0,0,0,0,False
35,AGRO-039,"Podridão-preta-das-raízes, Podridão-cinzenta-d...",exploratoria_menos_de_3_grupos,1,1,1,1,0,0,0,0,False
43,AGRO-047,Broca-das-axilas-da-soja,exploratoria_menos_de_3_grupos,1,1,1,1,0,0,0,0,False


## 4. Auditoria de vazamento e cobertura

Esta célula interrompe a execução se alguma origem, grupo ou hash exato atravessar partições. Ela também confere novamente os pares visuais confirmados cujas duas imagens foram mantidas. A ausência de vínculos manuais não comprova que duas fotos de catálogo sejam de plantas diferentes.

In [5]:
assert manifesto.groupby("grupo_particao")["particao"].nunique().eq(1).all()
assert manifesto.groupby("grupo_origem")["particao"].nunique().eq(1).all()
assert manifesto.loc[manifesto["sha256"].notna()].groupby("sha256")["particao"].nunique().eq(1).all()

por_id = manifesto.set_index("image_id")["particao"].to_dict()
for par in pares_visuais.itertuples(index=False):
    resultado = str(par.resultado_revisao).casefold()
    if resultado.startswith("mesma foto") or resultado.startswith("mesma fotografia"):
        if par.image_id_1 in por_id and par.image_id_2 in por_id:
            assert por_id[par.image_id_1] == por_id[par.image_id_2]

assert not auditoria["classes_candidatas_sem_cobertura"]
assert not auditoria["pendencias_da_ancoragem"]
assert resumo.loc[
    resumo["status_classe"].eq("exploratoria_menos_de_3_grupos"), "imagens_validacao"
].eq(0).all()
assert resumo.loc[
    resumo["status_classe"].eq("exploratoria_menos_de_3_grupos"), "imagens_teste"
].eq(0).all()
print("Auditoria estrutural aprovada: nenhum vazamento identificável entre partições.")
display(pd.crosstab(manifesto["fonte"], manifesto["particao"]))

Auditoria estrutural aprovada: nenhum vazamento identificável entre partições.


particao,teste,treino,validacao
fonte,,,
acervo_milho,3,9,3
acervo_soja,11,69,11
dataset_campo,268,1255,269
dataset_catalogo,205,954,205


## 5. Exportação

O manifesto inclui IDs, caminhos, rótulos candidatos, fonte, grupos, partição e status da classe. O resumo por classe informa imagens e grupos por partição. `auditoria.json` registra semente, proporções, hashes dos arquivos de entrada e ressalvas. Esses arquivos são **provisórios**; não constituem aprovação dos rótulos nem avaliação final do modelo.

In [6]:
PASTA_SAIDA = salvar_resultados(ROOT, manifesto, resumo, auditoria, {**proveniencia, **diagnostico_grupos})
for caminho in sorted(PASTA_SAIDA.iterdir()):
    print(caminho.relative_to(ROOT), f"({caminho.stat().st_size:,} bytes)")

assets\data\particionamento_T4_2\auditoria.json (1,039 bytes)
assets\data\particionamento_T4_2\manifesto_provisorio.csv (967,370 bytes)
assets\data\particionamento_T4_2\resumo_por_classe.csv (5,647 bytes)


## Próximos passos para T4.3

Confirme as classes finais e os vínculos de mesma planta, conclua a revisão visual pendente e **refaça** a divisão. Só então calcule a ponderação de classes com os exemplos originais **do treino**: `w_c = N / (K × n_c)`. Mantenha a amostragem natural inicialmente; não combine ponderação e reamostragem no mesmo experimento inicial. Validação e teste permanecem com imagens originais, sem aumento de dados e sem frequências usadas no cálculo dos pesos. Classes exploratórias não recebem métricas de validação ou teste.